# Wine Classification and Regression

## Learning Objectives

By the end of this notebook, students will be able to:

1.  **Prepare Data for Machine Learning**: Grasp the importance of splitting data into training, validation, and test sets, and understand the purpose of feature scaling.
2.  **Implement and Tune Classification Models**: Build, train, and evaluate three different classification models (K-Nearest Neighbors, Decision Trees, and Random Forests)
3.  **Understand the Bias-Variance Tradeoff**: Visualize and explain how model complexity affects model performance on training and validation data. Detect overfitting, and find the right model complexity for a given task.
4.  **Interpret Model Results**: Learn how to interpret the results of classification models. Display decision of a tree model and feature importance in Random Forests.

## 1. Setup: Importing Necessary Libraries

Before we begin, we need to import the libraries that will provide the tools for our analysis.

In [1]:
# Core libraries for data manipulation and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: Library for classic machine learning in Python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

from tqdm import tqdm

## 2. Data Loading and Initial Exploration

Load the dataset from `data.csv` file into a Pandas DataFrame. Similar to the previous notebook, we will perform initial data cleaning (handling missing values and duplicates).

In [2]:
wine_df = pd.read_csv('data.csv')

In [ ]:
wine_df.head()

## 3. Data Preprocessing

### 3.1. Data Cleaning
Remove any missing values and duplicates in the dataset.

In [ ]:
# Remove rows with missing values
wine_df = wine_df.____
# Remove rows with duplicate values
wine_df = wine_df.____

### 3.2. Feature and Target Separation
divide the dataset into features (`X`) and target variable (`y`). Then we will train a model to predict `y` based on `X`, we are trying to predict type of wine (red=0 or white=1) based on its chemical properties.
We will use the ``dataframe.drop()`` function which drops a row column from a dataframe (https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html).

In [ ]:
X = wine_df.____
y = wine_df____

### 3.3. Target Encoding
Encode the target variable `y` where red wine is represented as `0` and white wine as `1`.

In [ ]:
y.loc[____] = 0
y.loc[____] = 1
y = y.astype(int)

In [ ]:
print(X.shape, y.shape)

### 3.4. Feature Scaling
As seen in the previous tutorial, feature scaling is crucial for algorithms like KNN for instance if a feature is 100x larger than another, it will dominate the distance calculations which will bias the model. We will use *StandardScaler* from *sklearn.preprocessing* to scale the features removing the mean and scaling to unit variance (https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).

In [12]:
scaler = ____
X_scaled = scaler.fit_transform(X)

### 3.5. Data Splitting
To be able to objectively evaluate our models, we will split the dataset into training, validation, and test sets. We aim to have **60%** of the data for training, **20%** for validation, and **20%** for testing.
We will be using stratified sampling to ensure that the class distribution is maintained across all sets. The documentation for `train_test_split` can be found [here](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In [18]:
X_train, X_test, y_train, y_test = train_test_split(____)
X_train, X_val, y_train, y_val = train_test_split(____) # 0.25 x 0.8 = 0.2

In [ ]:
def add_percentage_labels(ax):
    total = sum([p.get_height() for p in ax.patches])
    for p in ax.patches:        
        percentage = '{:.1f}%'.format(100 * p.get_height() / total)
        x = p.get_x() + p.get_width() / 2
        y_height = p.get_height()
        ax.annotate(percentage, (x, y_height), ha='center', va='bottom')

plt.figure(figsize=(12, 4))

plt.subplot(1, 4, 1)
ax1 = y.value_counts().sort_index().plot(kind='bar', title='Original Dataset', color=['yellow', 'red'], edgecolor='black')
plt.xticks(ticks=[0, 1], labels=['White', 'Red'], rotation=0)
ax1.set_ylim(0, ax1.get_ylim()[1] * 1.1)
add_percentage_labels(ax1)

# Training Set
plt.subplot(1, 4, 2)
ax2 = y_train.value_counts().sort_index().plot(kind='bar', title='Training Set', color=['yellow', 'red'], edgecolor='black')
plt.xticks(ticks=[0, 1], labels=['White', 'Red'], rotation=0)
ax2.set_ylim(0, ax2.get_ylim()[1] * 1.1)
add_percentage_labels(ax2)

# Validation Set
plt.subplot(1, 4, 3)
ax3 = y_val.value_counts().sort_index().plot(kind='bar', title='Validation Set', color=['yellow', 'red'], edgecolor='black')
plt.xticks(ticks=[0, 1], labels=['White', 'Red'], rotation=0)
ax3.set_ylim(0, ax3.get_ylim()[1] * 1.1)
add_percentage_labels(ax3)

# Test Set
plt.subplot(1, 4, 4)
ax4 = y_test.value_counts().sort_index().plot(kind='bar', title='Test Set', color=['yellow', 'red'], edgecolor='black')
plt.xticks(ticks=[0, 1], labels=['White', 'Red'], rotation=0)
ax4.set_ylim(0, ax4.get_ylim()[1] * 1.1)
add_percentage_labels(ax4)

plt.tight_layout()
plt.show()

## 4. Model Training and Evaluation
Hyperparameter selection is a crucial step in building effective and efficient machine learning models. As when different dataset will require different model complexities to capture underlying patterns without overfitting or underfitting, we will explore how to select optimal hyperparameters for our models and for the task of classifying wine types.

### 4.1. K-Nearest Neighbors (KNN)
The first model we will explore is the **K-Nearest Neighbors (KNN)** algorithm. KNN is a straightforward, instance-based learning algorithm. Instead of learning a model with parameters, it memorizes the training dataset. To classify a new data point, it looks at the $k$ closest labeled examples in the feature space and assigns the class based on a majority vote among them.

* **Documentation:** [sklearn.neighbors.KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

**Task:**
The choice of $k$ significantly impacts performance. A small $k$ captures fine details but is sensitive to noise, while a large $k$ smoothes the decision boundary but may miss local patterns. Using the provided code template, iterate through the different neighbors to determine which $k$ yields the highest accuracy on the validation set.

In [ ]:
k_values = range(1, 45)
accuracies = []
for k in tqdm(k_values, desc="Evaluating KNN k values", colour='blue'):
    knn = KNeighborsClassifier(____)
    knn.fit(____, ____)
    y_pred = knn.predict(____)
    accuracy = accuracy_score(____, ____)
    accuracies.append(accuracy)
plt.figure(figsize=(10, 6))
plt.plot(k_values, accuracies, marker='o', linestyle='-')
plt.title('KNN Accuracy vs. k Value')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Validation Accuracy')
plt.xticks(k_values)
plt.grid(True)
plt.show()

#### Analysis of KNN Performance

In [ ]:
best_k = ____
knn_best = KNeighborsClassifier(____)
knn_best.fit(____, ____)
print(f"Best KNN model (k={best_k}) accuracy on Test Set: {knn_best.score(____, ____):.4f}")

### 4.2. Decision Tree Classifier
The second model is the **Decision Tree Classifier**. This algorithm builds a model in the form of a tree structure. It breaks down the dataset into smaller subsets by asking a series of True/False questions (decision nodes) until a final prediction can be made at the "leaf" nodes.

* **Documentation:** [sklearn.tree.DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)

**Task:**
Decision trees can easily "overfit" (memorize) the training data if they are allowed to grow too deep. Using the provided variable `tree_depth_values`, iterate through the range of integers from **1 to 50** to find the optimal `max_depth` for your model.

In [ ]:
depth_values = range(1, 50)
accuracies_dt = []
for depth in tqdm(depth_values, desc="Evaluating Tree Depths", colour='blue'):
    dt = DecisionTreeClassifier(____)
    dt.fit(____, ____)
    y_pred = dt.predict(____)
    accuracy = accuracy_score(____, ____)
    accuracies_dt.append(accuracy)
plt.figure(figsize=(10, 6))
plt.plot(depth_values, accuracies_dt, marker='o', linestyle='-')
plt.title('Decision Tree Accuracy vs. Max Depth')
plt.xlabel('Max Depth')
plt.ylabel('Validation Accuracy')
plt.xticks(depth_values)
plt.grid(True)
plt.show()

In [ ]:
best_depth = 4
dt_best = DecisionTreeClassifier(____)
dt_best.fit(____, ____)
print(f"Best Decision Tree model (depth={best_depth}) accuracy on Test Set: {dt_best.score(____, ____):.4f}")

#### Visualizing the Decision Tree

In [ ]:
plt.figure(figsize=(20,10))
plot_tree(dt_best, filled=True, feature_names=X.columns, class_names=['Red', 'White'])
plt.show()

### 4.3. Random Forest Classifier
The third model is the **Random Forest Classifier**. This is an "ensemble" learning method. Instead of relying on just one decision tree, it constructs several decision trees during training. For classification tasks, it outputs the class selected by the majority of the individual trees. This generally improves accuracy and controls overfitting.

* **Documentation:** [sklearn.ensemble.RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

**Task:**
The number of trees in the forest is a key hyperparameter. A higher number of trees increases model stability but also computation cost. Using the provided `n_estimators_values`, test the performance by varying the number of trees from **1 to 150** (with a step size of 5) to determine the best configuration.


In [ ]:
n_estimators_values = range(1, 150, 5)
accuracies_rf = []
for n in tqdm(n_estimators_values, desc="Evaluating n_estimators"):
    rf = RandomForestClassifier(____)
    rf.fit(____, ____)
    y_pred = rf.predict(____)
    accuracy = accuracy_score(____, ____)
    accuracies_rf.append(accuracy)
plt.figure(figsize=(10, 6))
plt.plot(n_estimators_values, accuracies_rf, marker='o', linestyle='-')
plt.title('Random Forest Accuracy vs. Number of Trees')
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Validation Accuracy')
plt.grid(True)
plt.show()

In [ ]:
best_n_estimators = 16
rf_best = RandomForestClassifier(____)
rf_best.fit(____, ____)
print(f"Best Random Forest model (n_estimators={best_n_estimators}) accuracy on Test Set: {rf_best.score(____, ____):.4f}")

#### Interpreting Random Forest: Feature Importance

In [ ]:
rf_feature_importances = pd.Series(rf_best.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(12, 6))
sns.barplot(x=rf_feature_importances, y=rf_feature_importances.index)
plt.title('Random Forest Feature Importances')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.show()

## 5. Model Comparison and Conclusion

We have tuned our three contenders: KNN, Decision Tree, and Random Forest. Now, it is time for the final benchmarking.

To truly understand which model is best, we cannot rely on accuracy alone. We need to visualize **how** they make mistakes. For this, we will use the **Confusion Matrix**.

**Task:**
1.  **Generate Predictions:** Use your optimized models (the ones found in Section 4) to predict the labels for the held-out `test` set.
2.  **Visualize:** Run the provided code to generate side-by-side confusion matrices.
3.  **The Verdict:** Compare the plots. Look at the off-diagonal numbers (the errors).
    * Which model has the fewest False Positives?
    * Which model seems most robust overall?

In [57]:
y_pred_knn = knn_best.predict(____)
y_pred_dt = dt_best.predict(____)
y_pred_rf = rf_best.predict(____)

In [ ]:
cm_knn = confusion_matrix(y_test, y_pred_knn)
cm_dt = confusion_matrix(y_test, y_pred_dt)
cm_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(18, 5))
plt.subplot(1, 3, 1)
ConfusionMatrixDisplay(cm_knn, display_labels=['Red', 'White']).plot(ax=plt.gca(), cmap='Blues', colorbar=False)
plt.title('KNN Confusion Matrix')
plt.subplot(1, 3, 2)
ConfusionMatrixDisplay(cm_dt, display_labels=['Red', 'White']).plot(ax=plt.gca(), cmap='Blues', colorbar=False)
plt.title('Decision Tree Confusion Matrix')
plt.subplot(1, 3, 3)
ConfusionMatrixDisplay(cm_rf, display_labels=['Red', 'White']).plot(ax=plt.gca(), cmap='Blues', colorbar=False)
plt.title('Random Forest Confusion Matrix')
plt.show()

### Conclusion
Now that we have visualized the results, let's interpret them. Look closely at the "off-diagonal" numbers (the errors) in the Confusion Matrices above.

**Answer the following questions:**

1.  **The Loser:** Which model had the most total errors?
2.  **The Winner:** Which model performed best overall?
3.  Imagine that **"White"** represents a **safe chemical** and **"Red"** represents a **toxic chemical**.
    * *Scenario:* If you predict a toxic chemical is safe (False Negative), people get hurt. If you predict a safe chemical is toxic (False Positive), you just waste money on testing.
    * Based on this scenario, which model is the safest choice?